In [ ]:
# ============================================================
# CELL 1: GPU Check & Environment Setup
# ============================================================
import torch
import os

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Total VRAM   : {total_mem:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️  No GPU found — using CPU (training will be slow)")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, random, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("✅ All imports successful.")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

# ── Paths ──────────────────────────────────────────────────
DATASET_PATH = Path("/kaggle/input/datasets/bapdesilva/betel-dataset-bp/Betel Leaf Image Dataset from Bangladesh/Augmented Images/Train")
AUGMENTED_PATH = Path("/kaggle/working/augmented_dataset")   # offline-augmented copy
FINAL_PATH     = Path("/kaggle/working/final_dataset")        # train/val split lives here

# ── Dataset ────────────────────────────────────────────────
TARGET_PER_CLASS = 2000   # augment up to this count before any training
VAL_SPLIT        = 0.20   # 80/20 train-val split
IMG_SIZE         = 224    # EfficientNetB0 native resolution

# ── Training hyper-parameters ──────────────────────────────
BATCH_SIZE      = 32
NUM_EPOCHS      = 30
LR_HEAD         = 1e-3    # Phase-1: only classifier head
LR_FINE         = 1e-4    # Phase-2: fine-tune unlocked layers
WEIGHT_DECAY    = 1e-4
PATIENCE        = 7       # early-stopping patience (per phase)
UNFREEZE_PCT    = 0.30    # fraction of backbone layers to unlock in Phase-2

CLASS_NAMES = ["Bacterial Leaf Disease", "Dried Leaf",
               "Fungal Brown Spot Disease", "Healthy Leaf"]
NUM_CLASSES = len(CLASS_NAMES)

print("✅ Configuration set.")
print(f"   Dataset path  : {DATASET_PATH}")
print(f"   Target/class  : {TARGET_PER_CLASS}")
print(f"   Val split     : {VAL_SPLIT*100:.0f}%")
print(f"   Device        : {DEVICE}")

In [ ]:
# ============================================================
# CELL 4: Offline Augmentation (run ONCE — before training)
# Creates a balanced dataset at AUGMENTED_PATH
# ============================================================

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

def augment_class_folder(src_folder: Path, dst_folder: Path, target: int):
    """
    Copy originals + generate synthetic images until we reach `target`.
    dst_folder is created fresh each run (idempotent).
    """
    dst_folder.mkdir(parents=True, exist_ok=True)

    # Collect original images
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
    originals = [p for p in src_folder.iterdir() if p.suffix.lower() in exts]

    if not originals:
        print(f"  ⚠️  No images found in {src_folder} — skipping.")
        return 0

    count = 0

    # 1. Copy originals first
    for p in originals:
        shutil.copy(p, dst_folder / p.name)
        count += 1

    # 2. Augment until target is reached
    idx = 0
    while count < target:
        src_img_path = originals[idx % len(originals)]
        img = Image.open(src_img_path).convert("RGB")
        aug_img = aug_transform(img)
        save_name = f"aug_{count:05d}.jpg"
        aug_img.save(dst_folder / save_name)
        count += 1
        idx += 1

    return count


print("=" * 55)
print("OFFLINE AUGMENTATION  (target = {} images/class)".format(TARGET_PER_CLASS))
print("=" * 55)

total_generated = 0
for cls in CLASS_NAMES:
    src = DATASET_PATH / cls
    dst = AUGMENTED_PATH / cls

    if not src.exists():
        print(f"  ❌ Folder not found: {src}")
        continue

    n_orig = len([f for f in src.iterdir()
                  if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tiff",".webp"}])
    print(f"\n  [{cls}]  originals: {n_orig}", end=" → ")

    n_final = augment_class_folder(src, dst, TARGET_PER_CLASS)
    print(f"total after augmentation: {n_final}")
    total_generated += n_final

print(f"\n✅ Augmentation complete.  Grand total images: {total_generated}")

In [ ]:
# ============================================================
# CELL 5: Train / Validation Split  (stratified, no augmentation)
# ============================================================

def stratified_split(src_root: Path, dst_root: Path, val_frac: float, seed: int = SEED):
    """
    Copies files from src_root/<class>/ into
    dst_root/train/<class>/ and dst_root/val/<class>/
    """
    random.seed(seed)
    summary = {}

    for cls_dir in sorted(src_root.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name
        all_imgs = sorted(cls_dir.glob("*.*"))
        random.shuffle(all_imgs)

        n_val   = int(len(all_imgs) * val_frac)
        val_imgs = all_imgs[:n_val]
        trn_imgs = all_imgs[n_val:]

        for split, imgs in [("train", trn_imgs), ("val", val_imgs)]:
            out_dir = dst_root / split / cls_name
            out_dir.mkdir(parents=True, exist_ok=True)
            for p in imgs:
                shutil.copy(p, out_dir / p.name)

        summary[cls_name] = {"train": len(trn_imgs), "val": len(val_imgs)}

    return summary


print("Splitting into train / val …")
split_info = stratified_split(AUGMENTED_PATH, FINAL_PATH, VAL_SPLIT)

print("\n{:<30} {:>8} {:>8}".format("Class", "Train", "Val"))
print("-" * 50)
for cls, counts in split_info.items():
    print("{:<30} {:>8} {:>8}".format(cls, counts["train"], counts["val"]))

total_train = sum(v["train"] for v in split_info.values())
total_val   = sum(v["val"]   for v in split_info.values())
print("-" * 50)
print("{:<30} {:>8} {:>8}".format("TOTAL", total_train, total_val))
print("\n✅ Split complete.")

In [ ]:
# ============================================================
# CELL 6: DataLoaders  (NO augmentation transforms — already done)
# ============================================================

# EfficientNet ImageNet normalisation
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.ImageFolder(FINAL_PATH / "train", transform=train_transform)
val_dataset   = datasets.ImageFolder(FINAL_PATH / "val",   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Class mapping      : {train_dataset.class_to_idx}")

In [ ]:
# ============================================================
# CELL 7: EfficientNetB0 Model with Custom Head
# ============================================================

def build_model(num_classes: int, device: torch.device) -> nn.Module:
    """
    EfficientNetB0 backbone + custom classification head.
    All backbone weights frozen initially (Phase-1 head training).
    """
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model   = efficientnet_b0(weights=weights)

    # ── Freeze entire backbone ──────────────────────────────
    for param in model.parameters():
        param.requires_grad = False

    # ── Replace classifier head ─────────────────────────────
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )

    return model.to(device)


model = build_model(NUM_CLASSES, DEVICE)

# Print parameter counts
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable (head)    : {trainable_params:,}")

In [ ]:
# ============================================================
# CELL 8: Training Utilities
# ============================================================

class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4, restore_best=True):
        self.patience     = patience
        self.min_delta    = min_delta
        self.restore_best = restore_best
        self.best_loss    = float("inf")
        self.counter      = 0
        self.best_weights = None
        self.stop         = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            if self.restore_best:
                self.best_weights = {k: v.cpu().clone()
                                     for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                if self.restore_best and self.best_weights:
                    model.load_state_dict(
                        {k: v.to(DEVICE) for k, v in self.best_weights.items()}
                    )


def run_epoch(model, loader, criterion, optimizer, phase="train"):
    is_train = (phase == "train")
    model.train() if is_train else model.eval()

    running_loss, running_correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)
            running_loss    += loss.item() * imgs.size(0)
            running_correct += (preds == labels).sum().item()
            total           += imgs.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = running_correct / total
    return epoch_loss, epoch_acc


def unfreeze_top_layers(model, unfreeze_pct=0.30):
    """Unfreeze the last `unfreeze_pct` fraction of backbone feature layers."""
    feature_layers = list(model.features.children())
    n_unlock       = max(1, int(len(feature_layers) * unfreeze_pct))
    layers_to_unlock = feature_layers[-n_unlock:]

    for layer in layers_to_unlock:
        for param in layer.parameters():
            param.requires_grad = True

    unlocked = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total    = sum(p.numel() for p in model.parameters())
    print(f"  Unlocked last {n_unlock}/{len(feature_layers)} feature blocks")
    print(f"  Trainable params: {unlocked:,} / {total:,} "
          f"({100*unlocked/total:.1f}%)")


print("✅ Training utilities ready.")

In [ ]:
# ============================================================
# CELL 9: Phase 1 — Train classifier head only (backbone frozen)
# ============================================================

PHASE1_EPOCHS = NUM_EPOCHS  # up to 30; early stopping will kick in
criterion     = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_p1  = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler_p1  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=PHASE1_EPOCHS, eta_min=1e-6
)
early_stop_p1 = EarlyStopping(patience=PATIENCE, restore_best=True)

history_p1 = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   []}

print("=" * 60)
print("PHASE 1: HEAD TRAINING  (backbone frozen)")
print("=" * 60)

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer_p1, "train")
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, None,          "val")
    scheduler_p1.step()

    history_p1["train_loss"].append(tr_loss)
    history_p1["train_acc"].append(tr_acc)
    history_p1["val_loss"].append(vl_loss)
    history_p1["val_acc"].append(vl_acc)

    elapsed = time.time() - t0
    print(f"Ep {epoch:02d}/{PHASE1_EPOCHS}  "
          f"T-loss:{tr_loss:.4f}  T-acc:{tr_acc:.4f}  |  "
          f"V-loss:{vl_loss:.4f}  V-acc:{vl_acc:.4f}  "
          f"[{elapsed:.1f}s]")

    early_stop_p1(vl_loss, model)
    if early_stop_p1.stop:
        print(f"\n⏹  Early stopping at epoch {epoch} "
              f"(best val-loss: {early_stop_p1.best_loss:.4f})")
        break

print("\n✅ Phase 1 complete.")
torch.save(model.state_dict(), "/kaggle/working/efficientnet_b0_phase1.pth")
print("   Checkpoint saved → efficientnet_b0_phase1.pth")

In [ ]:
# ============================================================
# CELL 10: Phase 2 — Fine-tune (partial backbone unlock)
# ============================================================

print("=" * 60)
print(f"PHASE 2: FINE-TUNING  (unlock top {UNFREEZE_PCT*100:.0f}% of backbone)")
print("=" * 60)

unfreeze_top_layers(model, unfreeze_pct=UNFREEZE_PCT)

# Differential LRs: lower for backbone, higher for head
param_groups = [
    {"params": [p for n, p in model.named_parameters()
                if "classifier" not in n and p.requires_grad],
     "lr": LR_FINE},
    {"params": model.classifier.parameters(),
     "lr": LR_FINE * 5},
]

PHASE2_EPOCHS = 40
optimizer_p2  = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler_p2  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p2, T_0=10, T_mult=1, eta_min=1e-7
)
early_stop_p2 = EarlyStopping(patience=PATIENCE, restore_best=True)

history_p2 = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   []}

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer_p2, "train")
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, None,          "val")
    scheduler_p2.step()

    history_p2["train_loss"].append(tr_loss)
    history_p2["train_acc"].append(tr_acc)
    history_p2["val_loss"].append(vl_loss)
    history_p2["val_acc"].append(vl_acc)

    elapsed = time.time() - t0
    print(f"Ep {epoch:02d}/{PHASE2_EPOCHS}  "
          f"T-loss:{tr_loss:.4f}  T-acc:{tr_acc:.4f}  |  "
          f"V-loss:{vl_loss:.4f}  V-acc:{vl_acc:.4f}  "
          f"[{elapsed:.1f}s]")

    early_stop_p2(vl_loss, model)
    if early_stop_p2.stop:
        print(f"\n⏹  Early stopping at epoch {epoch} "
              f"(best val-loss: {early_stop_p2.best_loss:.4f})")
        break

print("\n✅ Phase 2 complete.")
torch.save(model.state_dict(), "/kaggle/working/efficientnet_b0_final.pth")
print("   Final checkpoint saved → efficientnet_b0_final.pth")

In [ ]:
# ============================================================
# CELL 11: Accuracy & Loss Curves
# ============================================================

def plot_history(hist, title_prefix=""):
    epochs_p1 = len(hist["p1"]["train_loss"])
    epochs_p2 = len(hist["p2"]["train_loss"])

    train_loss = hist["p1"]["train_loss"] + hist["p2"]["train_loss"]
    val_loss   = hist["p1"]["val_loss"]   + hist["p2"]["val_loss"]
    train_acc  = hist["p1"]["train_acc"]  + hist["p2"]["train_acc"]
    val_acc    = hist["p1"]["val_acc"]    + hist["p2"]["val_acc"]
    ep_x = range(1, len(train_loss) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    phase2_start = epochs_p1 + 0.5

    for ax, metric, ylabel in zip(
        axes,
        [(train_loss, val_loss), (train_acc, val_acc)],
        ["Loss", "Accuracy"]
    ):
        tr_vals, vl_vals = metric
        ax.plot(ep_x, tr_vals, label="Train",      color="#2196F3", lw=2)
        ax.plot(ep_x, vl_vals, label="Validation", color="#FF5722", lw=2)
        ax.axvline(x=phase2_start, color="green", linestyle="--",
                   alpha=0.7, label="Phase 2 start")
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f"{title_prefix}{ylabel} Curve", fontsize=14, fontweight="bold")
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved → training_curves.png")


combined_history = {"p1": history_p1, "p2": history_p2}
plot_history(combined_history, title_prefix="Betel Leaf CNN — ")

In [ ]:
# ============================================================
# CELL 12: Classification Report
# ============================================================

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Map indices back to class names in correct order
idx_to_cls = {v: k for k, v in val_dataset.class_to_idx.items()}
target_names = [idx_to_cls[i] for i in range(NUM_CLASSES)]

print("=" * 65)
print("CLASSIFICATION REPORT")
print("=" * 65)
print(classification_report(all_labels, all_preds,
                             target_names=target_names, digits=4))

In [ ]:
# ============================================================
# CELL 13: Confusion Matrix
# ============================================================

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── Raw counts ─────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Predicted Label", fontsize=11)
axes[0].set_ylabel("True Label", fontsize=11)
axes[0].tick_params(axis="x", rotation=30)
axes[0].tick_params(axis="y", rotation=0)

# ── Normalised (row %) ──────────────────────────────────────
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Confusion Matrix (row-normalised)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Predicted Label", fontsize=11)
axes[1].set_ylabel("True Label", fontsize=11)
axes[1].tick_params(axis="x", rotation=30)
axes[1].tick_params(axis="y", rotation=0)

plt.suptitle("Betel Leaf Disease — EfficientNetB0", fontsize=16,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → confusion_matrix.png")

In [ ]:
# ============================================================
# CELL 14: Feature Extractor — hook penultimate layer
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from pathlib import Path

# Load the trained model
model.eval()
model.to(DEVICE)

# ── Hook to extract features before the classifier head ────
# EfficientNetB0: model.avgpool outputs (B, 1280, 1, 1)
# We flatten that to (B, 1280) as the feature vector

extracted_features = {}

def hook_fn(module, input, output):
    extracted_features["feat"] = output.squeeze(-1).squeeze(-1)  # (B, 1280)

# Register hook on adaptive average pool
hook_handle = model.avgpool.register_forward_hook(hook_fn)

print("✅ Feature extractor hook registered on model.avgpool")
print(f"   Feature dimensionality: 1280")

In [ ]:
# ============================================================
# CELL 15: Extract features from TRAINING set (per class)
# These are used to compute class-conditional Gaussians
# ============================================================

# Use same val_transform (no augmentation) on training set
train_feat_dataset = datasets.ImageFolder(
    FINAL_PATH / "train", transform=val_transform
)
train_feat_loader = DataLoader(
    train_feat_dataset, batch_size=64,
    shuffle=False, num_workers=2, pin_memory=True
)

idx_to_cls = {v: k for k, v in train_feat_dataset.class_to_idx.items()}
print(f"Class mapping: {train_feat_dataset.class_to_idx}\n")

# Collect features and labels
all_feats  = []
all_labels = []

model.eval()
with torch.no_grad():
    for imgs, labels in train_feat_loader:
        imgs = imgs.to(DEVICE)
        _    = model(imgs)                         # triggers hook
        feats = extracted_features["feat"].cpu()   # (B, 1280)
        all_feats.append(feats)
        all_labels.append(labels)

all_feats  = torch.cat(all_feats,  dim=0).numpy()  # (N, 1280)
all_labels = torch.cat(all_labels, dim=0).numpy()  # (N,)

print(f"Total training features: {all_feats.shape}")

# ── Compute per-class mean vectors ─────────────────────────
class_means = {}
class_feats  = {}

for cls_idx in range(NUM_CLASSES):
    mask = (all_labels == cls_idx)
    feats_c = all_feats[mask]                # (Nc, 1280)
    class_feats[cls_idx]  = feats_c
    class_means[cls_idx]  = feats_c.mean(axis=0)
    print(f"  Class {cls_idx} [{idx_to_cls[cls_idx]:<30}]: {feats_c.shape[0]} samples")

print("\n✅ Per-class means computed.")